In [1]:
import os
import pandas as pd
import pyarrow.parquet as pq

In [2]:
def read_data():
    year = 0
    while year not in [2018, 2019, 2020]:
        year = int(input("Choose the year that you'd like to process (2018, 2019, 2020): "))
        if year not in [2018, 2019, 2020]:
            print("Invalid year. Please enter 2018, 2019, or 2020.")
        elif year == 2018:
            cargo_df = pd.read_csv('silver//cargo_2018_silver.csv')
            header_df = pd.read_csv('silver//shipment_2018_silver.csv')
            tariff_df = pd.read_csv('silver//tariff_2018_silver.csv')
            shipper_df = pd.read_csv('silver//shipper_2018_silver.csv')
            countries_df = pd.read_csv('silver//countries_2018_silver.csv')
        elif year == 2019:
            cargo_df = pd.read_csv('silver//cargo_2019_silver.csv')
            header_df = pd.read_csv('silver//shipment_2019_silver.csv')
            tariff_df = pd.read_csv('silver//tariff_2019_silver.csv')
            shipper_df = pd.read_csv('silver//shipper_2019_silver.csv')
            countries_df = pd.read_csv('silver//countries_2019_silver.csv')
        elif year == 2020:
            cargo_df = pd.read_csv('silver//cargo_2020_silver.csv')
            header_df = pd.read_csv('silver//shipment_2020_silver.csv')
            tariff_df = pd.read_csv('silver//tariff_2020_silver.csv')
            shipper_df = pd.read_csv('silver//shipper_2020_silver.csv')
            countries_df = pd.read_csv('silver//countries_2020_silver.csv')
    return header_df, shipper_df, cargo_df, tariff_df, countries_df, year

header_df, shipper_df, cargo_df, tariff_df, countries_df, year = read_data()

In [ ]:
def normalize_join_keys(df, columns):
    df = df.copy()
    if 'identifier' in columns and 'identifier' in df.columns:
        df['identifier'] = pd.to_numeric(df['identifier'], errors='coerce')
    if 'container_number' in columns and 'container_number' in df.columns:
        df['container_number'] = df['container_number'].astype(str).str.strip().str.upper()
    return df


def join_tables(header_df, shipper_df, cargo_df, tariff_df):
    header_df = normalize_join_keys(header_df, ['identifier'])
    shipper_df = normalize_join_keys(shipper_df, ['identifier'])
    cargo_df = normalize_join_keys(cargo_df, ['identifier', 'container_number'])
    tariff_df = normalize_join_keys(tariff_df, ['identifier', 'container_number'])

    header_shipper = header_df.merge(shipper_df, on='identifier', how='left')

    cargo_tariff = cargo_df.merge(
        tariff_df,
        on=['identifier', 'container_number'],
        how='left',
        suffixes=('', '_tariff')
    )

    merged_df = header_shipper.merge(cargo_tariff, on='identifier', how='left', suffixes=('', '_cargo'))
    return merged_df

In [5]:
def compare_tariffs():
    years = [2018, 2019, 2020]
    frames = []
    for y in years:
        df = pd.read_csv(f'silver//tariff_{y}_silver.csv')
        df['year'] = y
        frames.append(df)
    return pd.concat(frames, ignore_index=True)


def compare_yearly_shipments():
    years = [2018, 2019, 2020]
    frames = []
    for y in years:
        cargo = pd.read_csv(f'silver//cargo_{y}_silver.csv')
        header = pd.read_csv(f'silver//shipment_{y}_silver.csv')
        tariff = pd.read_csv(f'silver//tariff_{y}_silver.csv')
        shipper = pd.read_csv(f'silver//shipper_{y}_silver.csv')

        merged = join_tables(header, shipper, cargo, tariff)
        merged['year'] = y
        frames.append(merged)
    return pd.concat(frames, ignore_index=True)

In [6]:
def validate_tariff_data(yearly_shipments):
    total_nulls = yearly_shipments.isnull().sum().sum()
    if total_nulls == 0:
        print("✓ Tariff DataFrame is clean.")
    else: 
        print("✗ Tariff DataFrame has issues.")

In [10]:
yearly_shipments = compare_yearly_shipments()
validate_tariff_data(yearly_shipments)

✗ Tariff DataFrame has issues.


In [11]:
yearly_shipments.head()

,identifier,carrier_code,vessel_name,vessel_country_code,port_of_unlading,estimated_arrival_date,actual_arrival_date,weight_combined,shipper_party_name,city,...,container_number,description_sequence_number,piece_count,description_text,description_sequence_number_tariff,harmonized_number,harmonized_value,harmonized_weight_combined,harmonized_weight_kg,year
0,201801010,NaN,NaN,NaN,NaN,NaN,NaN,NaN,JET FAST COMPANY LIMITED,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
1,201801011,NaN,NaN,NaN,NaN,NaN,NaN,NaN,UNION WONDERFUL MACHINERY LTD.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
2,201801012,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"SUMEEKO INDUSTRIES CO.,LTD.",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
3,201801013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,YUTY INDUSTRIES CO. LTD.,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
4,201801014,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"BE SOUND CO., LTD.",NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2018
